# 01a. Tiền xử lý dữ liệu văn bản

Khảo sát phân bố dữ liệu, xây dựng quy trình chuẩn hóa chuỗi ký tự, teencode và tách từ tiếng Việt cho bộ dữ liệu ViMMSD phục vụ mô hình ngôn ngữ PhoBERT.

In [1]:
# Thiết lập môi trường thực thi
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/meo225/Multimodal_Sarcasm_Detection.git"
REF = "main"
CODE_DIR = "/tmp/IE403"

def get_github_token():
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
ON_COLAB = (not ON_KAGGLE) and ("COLAB_RELEASE_TAG" in os.environ)
ON_CLOUD = ON_KAGGLE or ON_COLAB
if ON_CLOUD:
    if ON_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print("Thư mục làm việc:", os.getcwd())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.0 MB/s eta 0:00:00
Thư mục làm việc: /tmp/IE403


In [2]:
import json, re
from collections import Counter
import pandas as pd
import emoji

from vimmsd.data.preprocessing import clean_text, TextPreprocessor, TEENCODE
from vimmsd.utils.config import load_config

# Nạp cấu hình và dữ liệu tập huấn luyện
cfg = load_config("configs/base.yaml")
train_json_path = Path(cfg.paths.data_dir) / "vimmsd-train.json"

with open(train_json_path, encoding="utf-8") as f:
    raw_train = json.load(f)

df_train = pd.DataFrame(list(raw_train.values()))
print(f"Nạp thành công {len(df_train):,} mẫu dữ liệu từ {train_json_path.name}")
df_train.head()

Nạp thành công 10,805 mẫu dữ liệu từ vimmsd-train.json


,image,caption,label
0,8ae451edcd8ebf697f8763ece249115813149c55733bf8...,Cô ấy trên mạng vs cô ấy ngoài đời =))),multi-sarcasm
1,35370ffd6c791d6f8c4ab3dd4363ed468fab41e4824ee9...,Người tâm linh giao tiếp với người thực tế :))),not-sarcasm
2,316fdd1477725b9fb1a55015ac06b68b92b50bd4303e08...,Hình như Trăng hôm nay đẹp quá mọi người ạ! 😃 ...,multi-sarcasm
3,8a0f34e0e30e4e5cfb306933c1d25fa801a5da78646b59...,MỌI NGƯỜI NGHĨ SAO VỀ PHÁT BIỂU CỦA SHARK VIỆT...,not-sarcasm
4,e517a5e95d1065886a7c815e82fe254381d4f9f4b244d4...,2 tay hai nàng chứ việc gì phải lệ hai hàng,multi-sarcasm


## 1. Khảo sát đặc tính dữ liệu văn bản

Thống kê phân bố nhãn, độ dài văn bản và các dạng nhiễu thường gặp trên tập huấn luyện.

In [3]:
# Phân bố nhãn bài toán
label_dist = df_train["label"].value_counts()
label_pct = df_train["label"].value_counts(normalize=True).mul(100)
summary_labels = pd.DataFrame({"Số lượng": label_dist, "Tỷ lệ (%)": label_pct.round(2)})
print("Bảng phân bố nhãn:")
display(summary_labels)

# Thống kê độ dài văn bản
df_train["word_count"] = df_train["caption"].apply(lambda c: len(str(c).split()))
df_train["char_count"] = df_train["caption"].apply(lambda c: len(str(c)))

print("Thống kê độ dài:")
display(df_train[["word_count", "char_count"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(1))

Bảng phân bố nhãn:


,Số lượng,Tỷ lệ (%)
label,,
not-sarcasm,6062,56.10
multi-sarcasm,4224,39.09
image-sarcasm,442,4.09
text-sarcasm,77,0.71


Thống kê độ dài:


,word_count,char_count
count,10805.0,10805.0
mean,50.7,232.7
std,81.0,368.3
min,1.0,1.0
50%,15.0,69.0
75%,54.0,251.0
90%,165.6,763.8
95%,230.0,1059.0
99%,358.0,1637.0
max,846.0,3914.0


In [4]:
# Thống kê các dạng nhiễu mạng xã hội
captions = df_train["caption"].fillna("").tolist()

url_cnt = sum(1 for c in captions if re.search(r"https?://|www\.", c))
mention_cnt = sum(1 for c in captions if re.search(r"@\w+", c))
hash_cnt = sum(1 for c in captions if "#" in c)
html_cnt = sum(1 for c in captions if re.search(r"&[a-zA-Z]+;|&#\d+;", c))
repeat_cnt = sum(1 for c in captions if re.search(r"([a-zA-ZÀ-ỹ])\1{2,}", c, re.IGNORECASE))
emoji_cnt = sum(1 for c in captions if emoji.emoji_count(c) > 0)

noise_summary = pd.DataFrame([
    {"Đặc trưng": "Hashtag", "Số mẫu xuất hiện": hash_cnt, "Tỷ lệ (%)": hash_cnt / len(captions) * 100},
    {"Đặc trưng": "Emoji", "Số mẫu xuất hiện": emoji_cnt, "Tỷ lệ (%)": emoji_cnt / len(captions) * 100},
    {"Đặc trưng": "Ký tự lặp", "Số mẫu xuất hiện": repeat_cnt, "Tỷ lệ (%)": repeat_cnt / len(captions) * 100},
    {"Đặc trưng": "Thực thể HTML", "Số mẫu xuất hiện": html_cnt, "Tỷ lệ (%)": html_cnt / len(captions) * 100},
    {"Đặc trưng": "Đường dẫn URL", "Số mẫu xuất hiện": url_cnt, "Tỷ lệ (%)": url_cnt / len(captions) * 100},
    {"Đặc trưng": "Đề cập người dùng", "Số mẫu xuất hiện": mention_cnt, "Tỷ lệ (%)": mention_cnt / len(captions) * 100},
])
display(noise_summary.round(2))

,Đặc trưng,Số mẫu xuất hiện,Tỷ lệ (%)
0,Hashtag,2422,22.42
1,Emoji,3330,30.82
2,Ký tự lặp,472,4.37
3,Thực thể HTML,222,2.05
4,Đường dẫn URL,185,1.71
5,Đề cập người dùng,33,0.31


In [5]:
# Thống kê tần suất emoji
all_emojis = []
for c in captions:
    all_emojis.extend(emoji.distinct_emoji_list(c))

top_emojis = Counter(all_emojis).most_common(12)
emoji_analysis = pd.DataFrame([
    {"Emoji": em, "Mô tả emoji": emoji.demojize(em), "Tần suất": cnt}
    for em, cnt in top_emojis
])
print("Top 12 emoji xuất hiện nhiều nhất:")
display(emoji_analysis)

Top 12 emoji xuất hiện nhiều nhất:


,Emoji,Mô tả emoji,Tần suất
0,❤,:red_heart:,161
1,🔥,:fire:,158
2,🥰,:smiling_face_with_hearts:,126
3,❤️,:red_heart:,120
4,🥹,:face_holding_back_tears:,114
5,😍,:smiling_face_with_heart-eyes:,112
6,😎,:smiling_face_with_sunglasses:,110
7,🥲,:smiling_face_with_tear:,108
8,😂,:face_with_tears_of_joy:,100
9,🤣,:rolling_on_the_floor_laughing:,98


## 2. Đánh giá quy trình tiền xử lý văn bản

Kiểm tra kết quả làm sạch, chuẩn hóa teencode và tách từ tiếng Việt trên các mẫu thực tế.

In [6]:
# Đánh giá trên các mẫu đại diện
selected_indices = [0, 12, 42, 88, 105, 170, 310, 440, 852, 1029]
eval_rows = []

for idx in selected_indices:
    raw = captions[idx]
    cleaned_demojize = clean_text(raw, emoji_mode="demojize", word_segment_=True)
    cleaned_keep = clean_text(raw, emoji_mode="keep", word_segment_=True)
    eval_rows.append({
        "Index": idx,
        "Văn bản gốc": raw,
        "Tiền xử lý demojize và tách từ": cleaned_demojize,
        "Tiền xử lý giữ nguyên emoji": cleaned_keep,
    })

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(eval_rows))

,Index,Văn bản gốc,Tiền xử lý demojize và tách từ,Tiền xử lý giữ nguyên emoji
0,0,Cô ấy trên mạng vs cô ấy ngoài đời =))),Cô ấy trên mạng với cô ấy ngoài đời =))),Cô ấy trên mạng với cô ấy ngoài đời =)))
1,12,"Được biết, nạn nhân là em C.X.Đ. (sinh năm 2003), vừa cùng gia đình vào TP.HCM thăm người thân, đã lên lịch bay về nhưng không ngờ gặp sự việc đau lòng…\nTheo đó, lúc 21 giờ 36 phút ngày 27.4, C.X.Đ. chạy xe máy, hướng từ đường Trường Chinh về Chợ Cầu thì nghe tiếng người dân hô c.ư.ớp.\nEm lập tức đ.u.ổ.i theo xe máy của 2 nghi phạm. Trong lúc t.r.uy đ.u.ổi, xe máy của X.Đ. đã va chạm với xe máy nghi phạm, dẫn đến vụ việc thương tâm và em không qua khỏi ngay sau đó.\nTrong thời gian ngắn, TP.HCM xảy ra liên tiếp 2 vụ việc đau lòng khiến nạn nhân ra đi mãi mãi vì cố đu.ổ.i theo c.ư.ớ.p. Các em đều còn quá trẻ… 🥺","Được biết , nạn_nhân là em C.X.Đ. ( sinh năm 2003 ) , vừa cùng gia_đình vào TP.HCM thăm người_thân , đã lên lịch bay về nhưng không ngờ gặp sự_việc đau_lòng … Theo đó , lúc 21 giờ 36 phút ngày 27.4 , C.X.Đ. chạy xe_máy , hướng từ đường Trường_Chinh về Chợ_Cầu thì nghe tiếng người_dân hô c . ư . ớp . Em lập_tức đ . u . ổ . i theo xe_máy của 2 nghi_phạm . Trong lúc t.r.uy đ . u . ổi , xe_máy của X.Đ. đã va_chạm với xe_máy nghi_phạm , dẫn đến vụ_việc thương_tâm và em không qua khỏi ngay sau đó . Trong thời_gian ngắn , TP.HCM xảy ra liên_tiếp 2 vụ_việc đau_lòng khiến nạn_nhân ra đi mãi_mãi vì cố_đu . ổ . i theo c . ư . ớ . p . Các em đều còn quá trẻ … pleading face","Được biết , nạn_nhân là em C.X.Đ. ( sinh năm 2003 ) , vừa cùng gia_đình vào TP.HCM thăm người_thân , đã lên lịch bay về nhưng không ngờ gặp sự_việc đau_lòng … Theo đó , lúc 21 giờ 36 phút ngày 27.4 , C.X.Đ. chạy xe_máy , hướng từ đường Trường_Chinh về Chợ_Cầu thì nghe tiếng người_dân hô c . ư . ớp . Em lập_tức đ . u . ổ . i theo xe_máy của 2 nghi_phạm . Trong lúc t.r.uy đ . u . ổi , xe_máy của X.Đ. đã va_chạm với xe_máy nghi_phạm , dẫn đến vụ_việc thương_tâm và em không qua khỏi ngay sau đó . Trong thời_gian ngắn , TP.HCM xảy ra liên_tiếp 2 vụ_việc đau_lòng khiến nạn_nhân ra đi mãi_mãi vì cố_đu . ổ . i theo c . ư . ớ . p . Các em đều còn quá trẻ … 🥺"
2,42,Sấm sét kinh hoàng xảy ra tại Trung Quốc!\nĐang ẻ gặp bắn tỉa…,Sấm sét kinh_hoàng xảy ra tại Trung_Quốc !_Đang ẻ gặp bắn tỉa …,Sấm sét kinh_hoàng xảy ra tại Trung_Quốc !_Đang ẻ gặp bắn tỉa …
3,88,Vỏ quýt dày thì có móng tay nhọn 😗,Vỏ quýt dày thì có móng tay nhọn kissing face,Vỏ quýt dày thì có móng tay nhọn 😗
4,105,"AI TÌM HỘ EM CON DREAM 29E1-56.789 CÁI\nNgày 30/5, Phòng CSGT (Công an TP Hà Nội) cho biết, đã bàn giao chiếc xe máy bị mất cho anh Nguyễn Duy Thái (SN 1999), đang tạm trú tại phường Mỹ Đình 2, quận Nam Từ Liêm.\nĐáng chú ý đây đã là lần thứ 2 Thái bị nhảy xe mà được công an thành phố alo đến nhận lại xe. Anh chia sẻ xe của mình bị nhảy khoảng tháng 4/2022 ở xóm trọ phường Cổ Nhuế 2. Trước đó, tháng 12/2021, chiếc xe trên đã bị nhảy nhưng được công an quận Cậu Giấy tìm thấy và trao trả cho Thái.\nNhìn người mà lại ngẫm bản thân, chiếc Dream bố mẹ tặng hồi đỗ đại học lên Hà Nội bị luộc 4 năm nay mà sủi mất tăm. Bùn ghê ghớm!","AI TÌM HỘ_EM CON DREAM_29E1-56 . 789 CÁI Ngày 30/5 , Phòng CSGT ( Công_an TP Hà_Nội ) cho biết , đã bàn_giao chiếc xe_máy bị mất cho anh Nguyễn_Duy_Thái ( SN 1999 ) , đang tạm_trú tại phường Mỹ_Đình 2 , quận Nam Từ_Liêm . Đáng chú_ý đây đã là lần thứ 2 Thái bị nhảy xe mà được công_an thành_phố alo đến nhận lại xe . Anh chia_sẻ xe của mình bị nhảy khoảng tháng 4/2022 ở xóm trọ phường Cổ_Nhuế 2 . Trước đó , tháng 12/2021 , chiếc xe trên đã bị nhảy nhưng được công_an quận Cậu_Giấy tìm thấy và trao_trả cho Thái . Nhìn người mà_lại ngẫm bản_thân , chiếc Dream bố_mẹ tặng hồi đỗ đại_học lên Hà_Nội bị luộc 4 năm nay mà sủi mất_tăm . Bùn ghê_ghớm !","AI TÌM HỘ_EM CON DREAM_29E1-56 . 789 CÁI Ngày 30/5 , Phòng CSGT ( Công_an TP Hà_Nội ) cho biết , đã bàn_giao chiếc xe_máy bị mất cho anh Nguyễn_Duy_Thái ( SN 1999 ) , đang tạm_trú tại phường Mỹ_Đình 2 , quận Nam Từ_Liêm . Đáng ch

## 3. Kiểm tra tính toàn vẹn ngữ nghĩa trên các trường hợp đặc biệt

Xác thực khả năng bảo toàn các từ viết tắt chuyên môn và từ vựng đặc thù.

In [7]:
edge_cases = [
    ("Từ vựng đặc thù", "Ăn 2 kg sữa chua cay, sinh SN 1999, mua bảo hiểm Dr.G và biết PK game"),
    ("Teencode và ký tự lặp", "Phục vụ đỉnh chóp lun á, đợi có 2 tiếng thui à khum quạo tí nào ngonnnn 🐧"),
    ("Thực thể HTML, URL và mention", "Ad ơi cmt này đỉnh vch &amp; link https://fb.com #review @admin"),
    ("Dấu câu và emoji", "Đồ ăn ngon tuyệt vời... Ăn xong vô viện lun ak ??? !!! 🤡"),
]

for category, text in edge_cases:
    print(f"[{category}]")
    print("  Gốc      :", text)
    print("  Chuẩn hóa:", clean_text(text, emoji_mode="demojize", word_segment_=True))
    print()

[Từ vựng đặc thù]
  Gốc      : Ăn 2 kg sữa chua cay, sinh SN 1999, mua bảo hiểm Dr.G và biết PK game
  Chuẩn hóa: Ăn 2 kg sữa chua_cay , sinh SN 1999 , mua bảo_hiểm Dr._G và biết PK game

[Teencode và ký tự lặp]
  Gốc      : Phục vụ đỉnh chóp lun á, đợi có 2 tiếng thui à khum quạo tí nào ngonnnn 🐧
  Chuẩn hóa: Phục_vụ đỉnh chóp luôn á , đợi có 2 tiếng thôi à không quạo_tí nào ngon penguin

[Thực thể HTML, URL và mention]
  Gốc      : Ad ơi cmt này đỉnh vch &amp; link https://fb.com #review @admin
  Chuẩn hóa: Ad ơi bình_luận này đỉnh vch & link review

[Dấu câu và emoji]
  Gốc      : Đồ ăn ngon tuyệt vời... Ăn xong vô viện lun ak ??? !!! 🤡
  Chuẩn hóa: Đồ_ăn ngon tuyệt_vời ... Ăn xong vô_viện luôn à ? ? ? ! ! ! clown face



## 4. Đo lường hiệu năng và cơ chế lưu trữ đệm

Đánh giá tốc độ xử lý của TextPreprocessor khi áp dụng cơ chế lưu cache ra đĩa.

In [8]:
import time

preprocessor = TextPreprocessor.from_config(cfg.data.text)
benchmark_sample = captions[:1000]

# Thực thi lần đầu và tạo cache
t0 = time.time()
res_cold = preprocessor.process_all(benchmark_sample, cache_dir="data/processed")
t_cold = time.time() - t0

# Thực thi lần hai đọc từ cache
t1 = time.time()
res_warm = preprocessor.process_all(benchmark_sample, cache_dir="data/processed")
t_warm = time.time() - t1

print(f"Thời gian xử lý lần đầu: {t_cold:.3f} giây")
print(f"Thời gian nạp từ cache: {t_warm:.4f} giây")
print(f"Hệ số tăng tốc: {t_cold / max(t_warm, 1e-6):.1f}x")
assert res_cold == res_warm, "Dữ liệu cache không khớp"
print("Kiểm tra cache: Hợp lệ")

Thời gian xử lý lần đầu: 2.778 giây
Thời gian nạp từ cache: 0.0067 giây
Hệ số tăng tốc: 415.9x
Kiểm tra cache: Hợp lệ


## 5. Ghi cache

Chuẩn hóa mọi caption trong train, public test và private test (file nào có thì ghi). Kết quả là `01a_text_preprocessing.json` trong `paths.cache_dir` (trên Kaggle là `/kaggle/working/cache`). Save Version để giữ file này. Notebook ghép ảnh, split, train và test attach output đó, rồi đặt `data.text.cache_dir` tới thư mục `cache` đã attach.


In [ ]:
cache_dir = Path(cfg.paths.cache_dir)
preprocessor = TextPreprocessor.from_config(cfg.data.text)
data_dir = Path(cfg.paths.data_dir)
json_names = [cfg.data.train_json, cfg.data.get("public_test_json"), cfg.data.get("private_test_json")]

captions_all = []
used_files = []
for name in json_names:
    if not name:
        continue
    path = data_dir / name
    if not path.exists():
        print("bỏ qua, không thấy", path.name)
        continue
    raw = json.loads(path.read_text(encoding="utf-8"))
    items = raw.values() if isinstance(raw, dict) else raw
    captions_all.extend(str(item.get("caption", "")) for item in items)
    used_files.append(path.name)

preprocessor.process_all(captions_all, cache_dir=cache_dir)
cache_file = preprocessor._cache_file(cache_dir)
print(f"Đã chuẩn hóa {len(captions_all):,} caption từ: {', '.join(used_files)}")
print("Output:", cache_file)


## 6. Kết luận và cấu hình thực nghiệm

1. Chuẩn hóa chuỗi ký tự: Quy trình xử lý triệt để các dạng nhiễu mạng xã hội, teencode thông dụng và bảo toàn chính xác các từ vựng đặc thù.
2. Tách từ tiếng Việt: Định dạng nối từ ghép bằng dấu gạch dưới tương thích với bộ từ vựng của mô hình PhoBERT.
3. Cấu hình độ dài chuỗi: Phân vị độ dài p95 đạt 230 từ. Cấu hình max_length = 256 đảm bảo bao phủ hơn 96% độ dài mẫu dữ liệu.
4. Xử lý biểu tượng cảm xúc: Sử dụng chế độ demojize để chuyển đổi biểu tượng cảm xúc thành văn bản ngữ nghĩa.